# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanmustafa119/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [10]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face authentication successful.")

Hugging Face authentication successful.


In [11]:
import duckdb

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB HTTP support loaded.")

DuckDB HTTP support loaded.


In [12]:
con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured for DuckDB.")

Hugging Face token configured for DuckDB.


In [13]:
feature_query = """
SELECT
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    sessions_organic
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10000
"""

feature_df = con.sql(feature_query).df()

print("Feature vector shape:", feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (10000, 6)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes

| Feature | Meaning | Missing values | Available when? |
|---|---|---|---|
| `gsc_impressions` | Number of observed Google Search impressions for the content record. | Missing values are kept as missing and will be handled during preprocessing. | Available at the decision moment because it is an observed March 2026 measurement. |
| `gsc_clicks` | Number of observed Google Search clicks for the content record. | Missing values are kept as missing and will be handled during preprocessing. | Available at the decision moment because it is an observed March 2026 measurement. |
| `gsc_avg_position` | Observed average Google Search position. | Missing values are kept as missing and will be handled during preprocessing. | Available at the decision moment because it is an observed March 2026 measurement. |
| `ga4_pageviews` | Number of observed page views. | Missing values are kept as missing and will be handled during preprocessing. | Available at the decision moment because it is an observed March 2026 measurement. |
| `sessions_organic` | Number of observed organic sessions. | Missing values are kept as missing and will be handled during preprocessing. | Available at the decision moment because it is an observed March 2026 measurement. |

`content_hash_id` is used only to identify the content record and is not treated as a predictive feature.

These features describe information available during the March 2026 decision period. I will not use future-period outcomes as features.

In [14]:
print("Missing values by feature:")
print(feature_df.isna().sum())

Missing values by feature:
content_hash_id        0
gsc_impressions        0
gsc_clicks             0
gsc_avg_position       0
ga4_pageviews       8516
sessions_organic    8516
dtype: int64


In [15]:
feature_df.isna().sum()

,0
content_hash_id,0
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
ga4_pageviews,8516
sessions_organic,8516


### Missing-value observations

The March 2026 feature frame contains no missing values for the selected GSC features. However, `ga4_pageviews` and `sessions_organic` each have 8,516 missing values out of 10,000 rows.

I will not automatically treat these missing GA4 values as zero because missing data may represent unavailable data rather than a measured value of zero. The missing values will therefore be handled explicitly during preprocessing.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

I checked the feature vector for information that would not have been available at the March 2026 decision moment.

A key leakage risk is using future-period information, such as April clicks or a future decline indicator, as a March feature. These fields contain information from the outcome period and would not be available when the decision is made.

I deliberately tested a future feature in the earlier analysis. The leaked feature produced an accuracy of approximately 0.99, which is an artificially strong result caused by future information.

I therefore exclude future-period performance fields and label-derived fields from the final feature vector.

In [16]:
# Check the final feature columns for obvious future/leakage fields

print("Final feature columns:")
print(feature_df.columns.tolist())

future_or_label_terms = [
    "future",
    "april",
    "may",
    "june",
    "decline",
    "label",
    "target"
]

possible_leaks = [
    col for col in feature_df.columns
    if any(term in col.lower() for term in future_or_label_terms)
]

print("\nPossible leakage columns found:")
print(possible_leaks)

Final feature columns:
['content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'sessions_organic']

Possible leakage columns found:
[]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

- `april_clicks` — excluded because April is after the March decision period and would introduce future information.
- Future-period performance fields — excluded because they would not be available at the March decision moment.
- Future decline indicators — excluded because they are derived from the outcome being predicted and would cause label leakage.
- Client names — excluded for privacy and public-safe analysis.
- Raw URLs — excluded because they are not necessary for this public analysis.
- Private search queries — excluded because they are not necessary for the public analysis.

The final feature vector contains only information intended to be available at the decision moment.

## Self-check

Before you submit, confirm each line honestly:

- ✔️ Every section above is filled — markdown thinking AND the code that backs it
- ✔️ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✔️ No client names, URLs, or private queries anywhere
- ✔️ My claims use careful words: observed, measured, directional, decision-support
- ✔️ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.